# Ngrok API Demo (Google Colab)

This notebook allows me to run your FastAPI endpoint (`api.py`) inside Google Colab using Colab's GPU, and exposes it to the public internet using `ngrok` so I can test it from your local Mac (e.g., using Postman).

## 0a. Google Colab Setup (Step 1)
Run this cell FIRST to install Conda in Google Colab.
**Note:** Colab will automatically restart the kernel after this cell finishes. This is normal! Wait for it to reconnect before moving to Step 0b.

In [3]:
try:
    import google.colab
    !pip install -q condacolab
    import condacolab
    condacolab.install()
except ImportError:
    print("Not running in Colab. Skipping Conda setup.")

✨🍰✨ Everything looks OK!


## 0b. Mount Drive & Load Environment (Step 2)
After the kernel restarts, run this cell to mount your Google Drive, navigate to the project folder, and install all dependencies (including Unsloth and API packages).

In [22]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    %cd "/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor"

    print("\n--- Installing Environment ---")
    !conda env update -n base -f environment.yml

    print("\n--- Installing Colab Unsloth Drivers & API Requirements ---")
    !pip install pyngrok
    !pip install pandas
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install --no-deps "xformers<0.0.27" peft accelerate bitsandbytes
    !pip install fastapi uvicorn pyngrok nest_asyncio python-multipart librosa soundfile pydub silero-vad
except ImportError:
    print("Not running in Colab. Skipping mount and environment update.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor

--- Installing Environment ---
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: / - failed

SpecsConfigurationConflictError: Requested specs conflict with configured specs.
  requested specs: 
    - ffmpeg=8.0.1
    - pip
    - python=3.11.15
  pinned specs: 
    - cuda-version=12
    - python=3.12
    - python_abi=3.12[build=*cp312*]
Use 'conda config --show-sources' to look for 'pinned_specs' and 'track_features'
configuration parameters.  Pinned specs may also be defined in the file
/usr/local/conda-meta/pinned.



--- Installing Colab Unsloth Drivers & API Requirements ---
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-nu26fppm/unsloth_f

## 1. Start Ngrok Tunnel

In [23]:
import sys
!{sys.executable} -m pip install pyngrok nest_asyncio

In [29]:
from pyngrok import ngrok
import nest_asyncio

# 1. Set your ngrok auth token here
NGROK_AUTH_TOKEN = "7zWYNaLMSPRAe7sUjH7uV_56riAPqToRZVbCzeJuMFi"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 2. Open a tunnel on port 8000 (where uvicorn will run)
public_url = ngrok.connect(8000)
print(f"\n>>> PUBLIC API URL: {public_url.public_url} <<<\n")

# 3. Allow asyncio to nest so uvicorn can run inside the Colab event loop
nest_asyncio.apply()


>>> PUBLIC API URL: https://1c66-35-252-73-101.ngrok-free.app <<<



## 2. Run the API Server
This cell will run the FastAPI server and keep running until you stop it.
Once it says `Application startup complete`, open Postman or terminal on your Mac and send a POST request to:
`<PUBLIC_API_URL>/extract` with a file attached as form-data (key: `file`).

In [7]:
import sys
!{sys.executable} -m pip install librosa soundfile pydub silero-vad

In [30]:
import uvicorn
%cd "/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor"
!pwd
# Start the API server on port 8000
!python -m uvicorn api:app --host 0.0.0.0 --port 8000

/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor
/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W0523 07:33:25.170000 25587 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0523 07:33:25.203000 25587 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
🦥 Unsloth Z

t=2026-05-23T07:41:24+0000 lvl=warn msg="Stopping forwarder" name=http-8000-9b4dbf39-f938-4550-87af-bf172d89e00b acceptErr="failed to accept connection: Listener closed"
t=2026-05-23T07:41:24+0000 lvl=warn msg="Error restarting forwarder" name=http-8000-9b4dbf39-f938-4550-87af-bf172d89e00b err="failed to start tunnel: remote gone away"


INFO:     Shutting down
INFO:     Finished server process [25587]
ERROR:    Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/uvicorn/_compat.py", line 30, in asyncio_run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/asyncio/base_events.py", line 641, in run_until_complete
    self.run_forever()
  File "/usr/local/lib/python3.11/asyncio/base_events.py", line 608, in run_forever
    self._run_once()
  File "/usr/local/lib/python3.11/asyncio/base_events.py", line 1936, in _run_once
    handle._run()
  File "/usr/local/lib/python3.11/asyncio/events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
  File "/usr/local/lib/python3.11/site-packages/uvicorn/server.py", line 78, in serve
    with self.capture_signals():
  File "/usr/